# Athar wall damage model: Qwen3.5 fine-tuning on Colab

This notebook trains Athar's own damage-mapping model. It takes a rectified wall photo and answers with
strict JSON elements (`damage_type`, `geometry`, normalised `points`, `severity`, `damage_class`,
`confidence`) that the backend turns into the wall map, the treatment plan and the accuracy card.

* **Base model:** Qwen3.5 Small (Apache-2.0), natively multimodal. The 4B model is the default.
* **Method:** bf16 LoRA with Unsloth. 4-bit QLoRA is not used for Qwen3.5 because its quantisation error is
  higher than usual for this family; 4-bit appears only in the GGUF files exported for inference.
* **Where it runs:** training happens only here, on a Colab GPU (L4 22.5 GB by default, A100 when available).
  The 16 GB laptop is for inference only, through the GGUF export at the end.
* **Data:** two kinds, both described in `ml/data/README.md`:
  * expert-approved Athar wall maps (the main task), split by site so no site appears in two splits;
  * licence-checked public close-ups of damaged walls, fetched automatically: historic walls in Cairo,
    the Kasbah of Algiers and Darbhanga Fort, brick masonry, concrete crack sets and crack masks.
* **Measured twice:** the same held-out rows are scored on the base model before training and on the
  fine-tuned model after it, so the notebook ends with a before/after table you can show as it is.

Run the cells in order. With no approved maps exported yet, the notebook still trains and scores on the
public close-ups; the wall-map score appears once an export is uploaded.

## 1. GPU and profile

The profile is chosen from GPU memory: 40 GB or more trains the 9B model, about 22 GB trains the 4B model
(set `TRY_9B = True` to try 9B, which is at the edge of an L4), and less than 16 GB falls back to 2B.

In [1]:
TRY_9B = False

import torch

assert torch.cuda.is_available(), "Choose Runtime > Change runtime type and pick an L4 or A100 GPU."
gpu = torch.cuda.get_device_properties(0)
VRAM_GB = gpu.total_memory / 1024**3
if VRAM_GB >= 40:
    PROFILE = "qwen35_9b_a100"
elif TRY_9B and VRAM_GB >= 20:
    PROFILE = "qwen35_9b_l4"
elif VRAM_GB >= 16:
    PROFILE = "qwen35_4b_l4"
else:
    PROFILE = "qwen35_2b_small"
print(f"{gpu.name}: {VRAM_GB:.1f} GB, bf16 support {torch.cuda.is_bf16_supported()}, profile {PROFILE}")

NVIDIA L4: 22.0 GB, bf16 support True, profile qwen35_4b_l4


## 2. Drive and repository

Training outputs go to `MyDrive/athar-ml`. Downloaded public archives are copied to `MyDrive/athar-ml/raw`, so
a new runtime does not download them again; they are extracted on the runtime's local disk, which is much
faster than Drive for tens of thousands of small images. If you have an Athar training export, put it at
`MyDrive/athar-ml/athar-export.zip` (download it with `GET /api/v1/ml/training-export` using a model engineer
account, or run `python -m ml.data.export_athar` on a machine that can reach the server).

For a private repository, add a Colab secret named `GITHUB_TOKEN` with read access.

In [2]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/athar-ml")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path("/content/Athar")
REPO_URL = "https://github.com/Muath477/Athar.git"

try:
    token = userdata.get("GITHUB_TOKEN")
except userdata.SecretNotFoundError:
    token = None
clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", clone_url, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], check=True, capture_output=True, text=True).stdout)

Mounted at /content/drive
099e7b4 Merge pull request #8 from Muath477/diriyah-workflow



## 3. Pinned libraries

Qwen3.5 needs transformers v5. The versions are pinned in `ml/requirements-colab.txt` and match what
Unsloth 2026.11.6 accepts. unrar and 7-Zip are installed for the Özgenel archives, which are published as `.rar`.

In [3]:
!apt-get -qq install -y unrar p7zip-full > /dev/null
!pip install -q uv
!uv pip install --system -q -r ml/requirements-colab.txt
from importlib import metadata

for package in ("unsloth", "transformers", "trl", "peft", "torch", "huggingface_hub"):
    print(package, metadata.version(package))

unsloth 2026.11.6
transformers 5.17.0
trl 1.15.0
peft 0.21.3
torch 2.14.1+cu130
huggingface_hub 1.33.0


## 4. Public datasets

`ml.data.sources` is the catalogue: each source has its licence, page, how it is fetched (a direct link, the
Mendeley Data public API, or a Hugging Face dataset pinned to a commit) and how its folders map to Athar's
glossary (`crack`, `plaster_loss`, `salt_efflorescence`, or a sound surface). Classes outside the glossary,
such as mould or vegetation, are dropped rather than called sound.

The default sources all have an open licence (CC BY 4.0). Two are opt-in: `panam_city` (2.7 GB) and
`bd3_building_defects` (licence unclear, research use only). Set `INCLUDE_OPT_IN = True` to add them.

The organise step prints the label counts per source. Read them: a source whose folders did not match
prints few or no rows instead of wrong labels.

In [4]:
INCLUDE_OPT_IN = False

raw_on_drive = DRIVE_DIR / "raw"
raw_local = Path("ml/data/raw")


def copy_downloads(source_root, target_root):
    for path in source_root.rglob("*"):
        relative = path.relative_to(source_root)
        if not path.is_file() or "extracted" in relative.parts or path.suffix == ".part":
            continue
        target = target_root / relative
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)


raw_on_drive.mkdir(parents=True, exist_ok=True)
copy_downloads(raw_on_drive, raw_local)
!python -m ml.data.collect list
opt_in = "--all" if INCLUDE_OPT_IN else ""
!python -m ml.data.collect organise --keep-going {opt_in}
copy_downloads(raw_local, raw_on_drive)

historical_cracks_cairo    default  CC BY 4.0                https://data.mendeley.com/datasets/xfk99kpmj9/1
kasbah_algiers             default  CC BY 4.0                https://data.mendeley.com/datasets/zwshr65gnc/2
darbhanga_fort             default  CC BY 4.0                https://data.mendeley.com/datasets/s6t5nn73bm/1
brickwork_cracks           default  CC BY 4.0                https://zenodo.org/records/8014150
rfsat_masonry              default  CC BY 4.0                https://zenodo.org/records/6870108
sdnet2018                  default  CC BY 4.0                https://digitalcommons.usu.edu/all_datasets/48
ozgenel_classification     default  CC BY 4.0                https://data.mendeley.com/datasets/5y9wdsg2zt/2
ozgenel_segmentation       default  CC BY 4.0                https://data.mendeley.com/datasets/jwsn7tfbrp/1
ctcd_tunnel_cracks         default  CC BY 4.0                https://huggingface.co/datasets/shiweiluo99/tunnel-crack-segmentation-dataset
panam_city      

## 5. Athar wall maps and the training files

Unpacks the Athar export when it is on Drive, then writes chat-format files:

* `train/val/test.jsonl`: Athar maps split by site; weak labels only in training and never from a held-out site.
* `public_test.jsonl`: 10% of the public close-ups, chosen by a stable hash, never trained on.
* Public close-ups also fill the validation file while there are no Athar maps, so the best checkpoint can
  still be picked by validation loss.

In [5]:
export_zip = DRIVE_DIR / "athar-export.zip"
if export_zip.exists():
    !python -m ml.data.export_athar --archive {export_zip}
else:
    print(
        "No Athar export yet: training uses the public close-ups only, and the wall-map score waits for it."
    )
!python -m ml.data.build_dataset --public-cap 1500
print(Path("ml/data/build/report.md").read_text())

No Athar export yet: training uses the public close-ups only, and the wall-map score waits for it.
| split | examples | sites |
|---|---|---|
| train | 2722 | 0 |
| val | 158 | 0 |
| test | 0 | 0 |
| public_test | 312 | 0 |

| split | public close-ups | crack | sound |
|---|---|---|---|
| train | 2430 | 1226 | 1204 |
| val | 158 | 69 | 89 |
| test | 0 | 0 | 0 |
| public_test | 312 | 155 | 157 |

| split | examples | sites |
|---|---|---|
| train | 2722 | 0 |
| val | 158 | 0 |
| test | 0 | 0 |
| public_test | 312 | 0 |

| split | public close-ups | crack | sound |
|---|---|---|---|
| train | 2430 | 1226 | 1204 |
| val | 158 | 69 | 89 |
| test | 0 | 0 | 0 |
| public_test | 312 | 155 | 157 |



## 6. Model and a look at one training example

Loads the profile, checks it matches the GPU choice, and prints one rendered training conversation so the
chat template used in training can be compared with the one used at inference.

In [6]:
import json

from ml.train import train as athar_train
from ml.train.profiles import fallback_ladder, load_config, pick_profile

assert pick_profile(VRAM_GB, try_9b=TRY_9B) == PROFILE
config = load_config(PROFILE)
print(config)
print("Memory fallback ladder:")
for step in fallback_ladder(config):
    print(f"  {step.name}: image side {step.max_image_side}px, LoRA r={step.lora.r}")

sample = athar_train.read_rows(config.data.train)[0]
print(json.dumps(sample["messages"], ensure_ascii=False, indent=1)[:3000])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1118 22:02:12.232000 10941 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1118 22:02:12.339000 10941 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
TrainConfig(key='qwen35_4b_l4', name='qwen35-4b-l4', model_id='unsloth/Qwen3.5-4B', base_model='Qwen/Qwen3.5-4B', min_vram_gb=16, max_image_side=896, max_seq_length=4096, lora=LoraSettings(r=16, alpha=16, dropout=0.0, finetune_vision_layers=True, finetune_language_layers=True, finetune_attention_modules=True, finetune_mlp_modules=True), train=TrainSettings(epochs=3, learning_rate=0.0002, lr_scheduler='cosine', warmup_ratio=0.03, weight_decay=0.01, max_grad_norm=1.0, per_device_batch=2, grad_accum=4, optim='adamw_8bit', seed=3407), data=DataSettings(root='ml/data', train='ml/data/build/train.jsonl', val='ml/data/build/val.jsonl', test='ml/data/build/test.js

## 7. Score the base model first

Before any training, the base model answers the held-out rows: up to `EVAL_LIMIT` public close-ups and every
Athar test map. A fresh LoRA adapter starts at zero, so this is the base model's own answer. General
vision-language models are weak at naming deterioration patterns (the RED.AI study measured 42% for GPT-4o on
stone), so this number is the honest starting point the fine-tuned model has to beat.

In [ ]:
import hashlib
import json
from pathlib import Path

from PIL import Image

from ml.eval.evaluate import evaluate_rows
from ml.eval.tags import comparison_markdown, evaluate_tags

EVAL_LIMIT = 300
DATA_ROOT = Path(athar_train.resolve(config.data.root))


def image_path(row):
    prompt = row["messages"][:-1]
    part = next(p for p in prompt[1]["content"] if p["type"] == "image")
    return DATA_ROOT / part["image"]


def readable(row):
    path = image_path(row)
    if "__MACOSX" in path.parts or path.name.startswith("._"):
        return False
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception:
        return False
    return True


def stable_sample(rows, limit):
    ordered = sorted(rows, key=lambda row: hashlib.sha256(row["id"].encode()).hexdigest())
    return [row for row in ordered if readable(row)][:limit]


public_eval = stable_sample(athar_train.read_rows(config.data.public_test), EVAL_LIMIT)
wall_eval = [row for row in athar_train.read_rows(config.data.test) if row["task"] == "wall_map" and readable(row)]
print(f"Held-out public close-ups: {len(public_eval)}, Athar test maps: {len(wall_eval)}")

base = athar_train.load_untrained(config)
before = {
    "public": evaluate_tags(public_eval, athar_train.predict(base, public_eval)),
    "walls": evaluate_rows(wall_eval, athar_train.predict(base, wall_eval)) if wall_eval else None,
}
del base
athar_train.free_gpu()
print(json.dumps({k: v for k, v in before["public"].items() if k != "by_source"}, indent=1))

Held-out public close-ups: 229, Athar test maps: 0
==((====))==  Unsloth 2026.11.6: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [ ]:
!find /content/Athar/ml/data/raw -name "__MACOSX" -type d -prune -exec rm -rf {} + ; find /content/Athar/ml/data/raw -name "._*" -delete
!cd /content/Athar && python -m ml.data.collect organise --keep-going && python -m ml.data.build_dataset --public-cap 1500

## 8. Training

Settings come from `ml/configs/<profile>.yaml`:

| setting | 4B on L4 (default) | 9B on L4 (edge) | 9B on A100 |
|---|---|---|---|
| precision | bf16 LoRA | bf16 LoRA | bf16 LoRA |
| LoRA r / alpha | 16 / 16 | 16 / 16 | 16 / 16 |
| vision layers | trained | frozen | trained |
| image side cap | 896 px | 672 px | 896 px |
| max sequence | 4096 | 2048 | 4096 |
| batch x accumulation | 2 x 4 | 1 x 8 | 2 x 4 |
| learning rate | 2e-4 cosine, 3% warm-up | 1e-4 cosine | 1e-4 cosine |
| epochs | 3, evaluation every epoch, best by eval loss | same | same |
| optimiser | AdamW 8-bit, weight decay 0.01, clip 1.0 | same | same |
| gradient checkpointing | `"unsloth"` | `"unsloth"` | `"unsloth"` |

Unsloth measures bf16 LoRA at about 10 GB for 4B and 22 GB for 9B before image tokens, so 9B on a 22.5 GB L4
sits at the limit. If memory runs out, training steps down on its own: smaller images first, then LoRA r=8,
then the next smaller model. An A100 is the safe choice for 9B.

Watch the training loss for the first 20 steps: it should fall quickly because the answer format is fixed.
A flat loss usually means the learning rate is too low; spikes mean it is too high.

In [ ]:
torch.cuda.reset_peak_memory_stats()
trained = athar_train.train_with_fallback(config)
config = trained.config
print(f"Trained {config.name}; peak GPU memory {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB")

In [ ]:
history = [entry for entry in trained.trainer.state.log_history if "eval_loss" in entry]
for entry in history:
    print(f"epoch {entry['epoch']:.0f}: eval loss {entry['eval_loss']:.4f}")

## 9. Save the adapter and the merged 16-bit model

In [ ]:
adapter_dir = athar_train.save_adapter(trained)
merged_dir = athar_train.save_merged(trained)
run_dir = DRIVE_DIR / config.name
shutil.copytree(adapter_dir, run_dir / "adapter", dirs_exist_ok=True)
print(f"Adapter copied to {run_dir / 'adapter'}; merged model at {merged_dir}")

## 10. Score the fine-tuned model on the same rows

The public table compares the base and fine-tuned model on the same held-out close-ups. The wall-map report
uses the same cell matching as the product's accuracy card (each element is rasterised onto the wall's grid,
then `score_maps` from the backend compares cells), plus the JSON validity rate, length and area errors, and a
comparison with the detector that produced each test map's first draft.

In [ ]:
from IPython.display import Markdown, display

from ml.eval.evaluate import markdown

after_public_predictions = athar_train.predict(trained, public_eval)
after = {"public": evaluate_tags(public_eval, after_public_predictions), "walls": None}
display(Markdown(comparison_markdown(before["public"], after["public"])))

if wall_eval:
    wall_predictions = athar_train.predict(trained, wall_eval)
    after["walls"] = evaluate_rows(wall_eval, wall_predictions)
    (run_dir / "predictions.jsonl").write_text(
        "".join(json.dumps(p, ensure_ascii=False) + "\n" for p in wall_predictions)
    )
    display(Markdown(markdown(after["walls"])))
else:
    print("No Athar test maps yet: export approved maps and re-run to get the wall-map score.")

(run_dir / "metrics.json").write_text(
    json.dumps({"before": before, "after": after}, ensure_ascii=False, indent=2)
)

## 11. GGUF for the 16 GB laptop

Converts the merged model with llama.cpp into a text GGUF and a separate `mmproj` vision projector, then
quantises to Q4_K_M (default for the laptop) and Q8_0. Both files are needed by `llama-server`. See
`ml/serve/README.md` for running them.

In [ ]:
LLAMA_CPP = Path("/content/llama.cpp")
GGUF_DIR = run_dir / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)
if not LLAMA_CPP.exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP}
!uv pip install --system -q {LLAMA_CPP}/gguf-py
!cmake -S {LLAMA_CPP} -B {LLAMA_CPP}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build {LLAMA_CPP}/build --target llama-quantize -j 8 > /dev/null

f16 = GGUF_DIR / "athar-damage-F16.gguf"
!python {LLAMA_CPP}/convert_hf_to_gguf.py {merged_dir} --outfile {f16} --outtype f16
!python {LLAMA_CPP}/convert_hf_to_gguf.py {merged_dir} --mmproj --outfile {GGUF_DIR}/mmproj-athar-damage-F16.gguf --outtype f16
for quant in ("Q4_K_M", "Q8_0"):
    !{LLAMA_CPP}/build/bin/llama-quantize {f16} {GGUF_DIR}/athar-damage-{quant}.gguf {quant}
for path in sorted(GGUF_DIR.glob("*.gguf")):
    print(f"{path.name}: {path.stat().st_size / 1024**3:.2f} GB")

## 12. Optional: push to the Hugging Face Hub

Set `PUSH_TO_HUB = True` and add a Colab secret `HF_TOKEN` with write access. Keep the repository private:
the weights were trained on site photos owned by the heritage authority.

In [ ]:
PUSH_TO_HUB = False
HUB_REPO = "your-account/athar-damage-qwen35"

if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HUB_REPO, private=True, exist_ok=True)
    api.upload_folder(repo_id=HUB_REPO, folder_path=str(run_dir / "adapter"), path_in_repo="adapter")
    api.upload_folder(
        repo_id=HUB_REPO,
        folder_path=str(GGUF_DIR),
        path_in_repo="gguf",
        allow_patterns=["*Q4_K_M*", "*Q8_0*", "mmproj*"],
    )

## 13. Register the version in Athar

Post this body to `POST /api/v1/ml/models` as a model engineer, then activate it from
`/dashboard/models` after comparing its metrics with the active version. The `name` must match the model
name the inference server answers to (`--alias` in llama-server, `--served-model-name` in vLLM).

In [ ]:
VERSION = "1"
wall_metrics = after["walls"] or {}
registration = {
    "name": f"athar-damage-v{VERSION}",
    "version": VERSION,
    "base_model": config.base_model,
    "adapter_uri": f"hf://{HUB_REPO}/adapter" if PUSH_TO_HUB else str(run_dir / "adapter"),
    "metrics": {
        **{
            key: wall_metrics[key]
            for key in ("json_valid_rate", "macro_f1", "micro_f1", "damaged_vs_sound_accuracy", "test_maps")
            if wall_metrics.get(key) is not None
        },
        "public_exact_match_before": before["public"]["exact_match"],
        "public_exact_match_after": after["public"]["exact_match"],
        "public_examples": after["public"]["examples"],
    },
}
print(json.dumps(registration, ensure_ascii=False, indent=2))